In [6]:
from dataclasses import dataclass
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from loguru import logger
from typing import Annotated, Literal
from langgraph.runtime import Runtime
from langgraph.graph.message import MessagesState
from langchain.messages import SystemMessage,HumanMessage, AIMessage, ToolMessage
from langgraph.store.postgres import PostgresStore
from langgraph.checkpoint.postgres import PostgresSaver


# 1.构建大语言模型
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import os

from openpyxl.styles.builtins import output

load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {
            "type": "disabled"
        }
    }
)

# 2.定义运行时的环境上下文
@dataclass
class UserContext:
    username: str
    membership_level: str

# 3.定义状态
class OverAllState(MessagesState):
    user_input: str
    output: str

# 4.定义结点
def llm_node(state: OverAllState, runtime: Runtime[UserContext]) -> OverAllState:
    # 1.获取环境上下文，判断当前的用户等级
    runtime_context = runtime.context

    if runtime_context:
        level = runtime_context.membership_level
        username = runtime_context.username
        logger.info(f"当前用户：{username},会员等级:{level}")

        if level == "VIP":
            system_prompt = f"你是高级客户助理，当前VIP用户是{username},请使用尊称'您',同时语气要热情周到，回复的末尾加上'VIP金牌服务'"
        else:
            system_prompt = f"你是普通客户助理，当前用户是{username},简洁回复即可,不准用'您'"
    else:
        system_prompt = f"你是普通客户助理,简洁回复即可"

    user_input = state["user_input"]
    messages = state.get("messages", [])
    output = model.invoke([SystemMessage(content = system_prompt)] + messages + [HumanMessage(content = user_input)])

    return {
        "messages": messages,
        "output": output.content
    }


# 5.构建图
builder = StateGraph(state_schema = OverAllState, context_schema = UserContext)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

# 第一次调用 传入VIP上下文用户
res1 = graph.invoke(
    {"user_input":"你好，帮我查一下最近有什么优惠活动？"},
    context = UserContext(username = "江来", membership_level = "VIP")
)
print(res1)

2026-10-27 21:57:20.119 | INFO     | __main__:llm_node:52 - 当前用户：江来,会员等级:VIP


{'messages': [], 'user_input': '你好，帮我查一下最近有什么优惠活动？', 'output': '您好！非常感谢您的咨询。关于最近的优惠活动，我马上为您查询，请稍等片刻哦～  \n目前已为您找到以下活动：  \n1. **VIP专享满减券**：满1000元减150元（有效期至本月底）。  \n2. **积分翻倍活动**：本周内消费可获得双倍积分。  \n3. **限时秒杀**：每日上午10点开放部分商品5折抢购。  \n您看是否需要我为您详细说明某一项活动？VIP金牌服务'}


In [7]:
# 第二次调用 传入普通上下文用户
res2 = graph.invoke(
    {"user_input":"你好，帮我查一下最近有什么优惠活动？"},
    context = UserContext(username = "江来", membership_level = "普通用户")
)
print(res2)

2026-10-27 21:57:23.271 | INFO     | __main__:llm_node:52 - 当前用户：江来,会员等级:普通用户


{'messages': [], 'user_input': '你好，帮我查一下最近有什么优惠活动？', 'output': '你好江来，最近有满减和折扣活动，具体要看你想了解哪类商品。'}
